# 3: rioxarray — Geospatial xarray

Open Cloud-Optimized GeoTIFFs from STAC as **xarray** objects, then clip and reproject with the `.rio` accessor.

This notebook uses two collections on Microsoft Planetary Computer:

- **`cop-dem-glo-30`** — Copernicus DEM, ~30 m elevation (replaces a local DEM GeoTIFF)
- **`sentinel-2-l2a`** — one red-band scene, so two rasters can be aligned

ERA5-Land daily precipitation/temperature GeoTIFFs are **not** published as COGs on that catalog. The closest climate collection is `era5-pds` (Zarr, not GeoTIFF) and needs extra packages (`zarr`, `adlfs`). Elevation is the raster that stays a signed COG.

**Dependencies:** `rioxarray`, `xarray`, `pystac-client`, `planetary-computer`, `matplotlib`

## What rioxarray adds

xarray does not know about CRS or geotransforms. **rioxarray** adds the `.rio` accessor:

- `open_rasterio(url)` — COG/GeoTIFF → `DataArray` with `band`, `y`, `x`
- `.rio.crs` / `.rio.bounds()` / `.rio.resolution()` — spatial metadata
- `.rio.clip_box()` / `.rio.reproject()` / `.rio.reproject_match()` — spatial ops
- `.rio.to_raster()` — write a GeoTIFF

`open_rasterio` on a COG URL reads the **header** first. Pixels load when a clip, plot, or write needs them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rioxarray
import pystac_client
import planetary_computer
from pathlib import Path

## Search and sign

Same study area as Notebooks 1–2. `CLIP` sits inside one Copernicus DEM 1° tile (`N16` / `E076`), so a single COG covers the window.

In [ ]:
# Northern Karnataka, India (same study area as Notebook 1)
# bbox order is always [min_lon, min_lat, max_lon, max_lat]
BBOX = [75.62, 16.20, 77.29, 17.47]
CLIP = (76.40, 16.75, 76.55, 16.90)  # west, south, east, north

# Metadata only — no pixels yet
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

# Copernicus DEM GLO-30: one ~1° COG tile per item
dem_search = catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=list(CLIP),
    max_items=1,
)
dem_item = planetary_computer.sign(list(dem_search.items())[0])
dem_url = dem_item.assets["data"].href  # elevation COG

# Sentinel-2 L2A: one low-cloud scene over the same window
s2_search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=list(CLIP),
    datetime="2023-06-01/2023-06-30",
    max_items=1,
    query={"eo:cloud_cover": {"lt": 10}},
)
s2_item = planetary_computer.sign(list(s2_search.items())[0])
s2_url = s2_item.assets["B04"].href  # red band COG

print("DEM item:", dem_item.id)
print("S2 item:", s2_item.id)

## Open the DEM

`open_rasterio` returns a `DataArray` with dimensions `(band, y, x)` and spatial metadata on `.rio`.

In [ ]:
# Header-only open: CRS, size, and transform — not the full 1° tile
dem = rioxarray.open_rasterio(dem_url, chunks={"x": 1024, "y": 1024})

print("Dimensions:", dem.dims)
print("Shape:", dem.shape)
print("CRS:", dem.rio.crs)
print("Bounds:", dem.rio.bounds())
print("Resolution:", dem.rio.resolution())  # degrees for this DEM (EPSG:4326)
dem

## Drop the band dimension

A single-band raster still arrives with `band=1`. Squeeze it so indexing is just `y`, `x`.

In [ ]:
dem = dem.squeeze("band", drop=True)
print("Dimensions after squeeze:", dem.dims)

## Clip to a bounding box

`clip_box` streams only the window that intersects `CLIP`. Pass `crs="EPSG:4326"` because the box is lon/lat (the DEM is already 4326; Sentinel-2 later is UTM).

In [ ]:
west, south, east, north = CLIP

# Reads only the intersecting COG window — not the full 3600×3600 DEM tile
# .load() pulls that small window into memory (541×541 here)
dem_clip = dem.rio.clip_box(
    minx=west, miny=south, maxx=east, maxy=north, crs="EPSG:4326"
).load()
print("Clipped shape:", dem_clip.shape)
print("Clipped bounds:", dem_clip.rio.bounds())

## Plot elevation

In [ ]:
# terrain colormap; colorbar is the legend; axis("off") hides x/y coordinates
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dem_clip.values, cmap="terrain")
ax.set_title("Copernicus DEM — CLIP window")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, label="Elevation (m)")
plt.tight_layout()
plt.show()

## Reproject to UTM

Northern Karnataka is **UTM zone 43N** (`EPSG:32643`). After reprojection, resolution is in metres instead of degrees.

In [ ]:
# Warp the clipped DEM onto a metre grid (zone 43N)
dem_utm = dem_clip.rio.reproject("EPSG:32643")

print("CRS:", dem_utm.rio.crs)
print("Resolution (m):", dem_utm.rio.resolution())
print("Shape:", dem_utm.shape)

## Align two rasters (`reproject_match`)

The DEM is ~30 m / EPSG:4326. Sentinel-2 B04 is 10 m / UTM. They cannot be added or overlaid until they share a grid.

`reproject_match(other)` warps the DEM onto the Sentinel-2 pixel grid (same CRS, bounds, and resolution).

In [ ]:
# Open B04 (UTM, 10 m) and clip to the same lon/lat window
s2 = rioxarray.open_rasterio(s2_url, chunks={"x": 1024, "y": 1024}).squeeze(
    "band", drop=True
)
s2_clip = s2.rio.clip_box(
    minx=west, miny=south, maxx=east, maxy=north, crs="EPSG:4326"
).load()

# Warp DEM onto the S2 grid so pixels line up 1:1
dem_on_s2 = dem_clip.rio.reproject_match(s2_clip)

print("S2 CRS / shape:", s2_clip.rio.crs, s2_clip.shape)
print("DEM-on-S2 CRS / shape:", dem_on_s2.rio.crs, dem_on_s2.shape)

In [ ]:
# Side-by-side: same window, same grid, different sensors
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(dem_on_s2.values, cmap="terrain")
axes[0].set_title("DEM on Sentinel-2 grid")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="Elevation (m)")

# B04 is uint16 (reflectance × 10000); stretch 2–98% so land is visible
valid = s2_clip.values[s2_clip.values > 0]
vmin, vmax = np.percentile(valid, (2, 98)) if valid.size else (0, 1)
im1 = axes[1].imshow(s2_clip.values, cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("Sentinel-2 B04")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="DN")

plt.tight_layout()
plt.show()

## Open with chunking (Dask-backed)

`chunks=` keeps the array **lazy**. Useful for large rasters and the next lesson (Dask).

In [ ]:
# Same URL, but pixels stay as a Dask array until .compute() or a plot
dem_chunked = rioxarray.open_rasterio(dem_url, chunks={"x": 1024, "y": 1024})
print("Chunks:", dem_chunked.chunks)
print("Dask?", hasattr(dem_chunked.data, "compute"))

## Write the clipped DEM

In [ ]:
# Write only the CLIP window (not the full 1° tile)
out_path = Path("output/dem_clip.tif")
out_path.parent.mkdir(parents=True, exist_ok=True)

dem_clip.rio.to_raster(out_path, compress="deflate", tiled=True)
print("Wrote", out_path.resolve())